In [14]:
import numpy as np
import math
import constants as c

# Constants
elementary_charge = 1.602e-19  # Charge of an electron (C)
k_coulomb = 8.988e9  # Coulomb's constant (N·m²/C²)
trap_coefficient = 1e3  # Coefficient for quadratic trap potential
mass_dm = 1.0e-27  # Mass of dark matter particle (kg)
hbar=1.055e-34,# reduced Planck constant

# Time step and duration
step = 1e-6  # Small time step for numerical integration
t = 0
time_limit = 0.10  # Stop condition

#Itai's Constants
bik=np.array([[1/np.sqrt(3),1/np.sqrt(3),1/np.sqrt(3)],[1/np.sqrt(2),0,-1/np.sqrt(2)],[1/np.sqrt(6),-2/np.sqrt(6),1/np.sqrt(6)]])
pi=np.pi
seconds=10**(12)
meter=seconds/(3*(10**8))
micrometer=meter/10**6
um=micrometer
omegavec=np.array([2.817,2.942,3.03])*2*pi*(10**6)/seconds
ms=seconds/1000
cm=meter/100
deltaxions=3.7*micrometer
eV=1.51927*10**15/seconds
kelvin=0.0000861733*eV
GeV=10**9*eV
mYb=173*0.9*GeV
temperature=300*kelvin
alpha=1/137.035999
x10=0*um
y10=89.1522*um
z10=-4.3887*um
x20=0
y20=89.1396*um
z20=0
x30=0
y30=89.1522*um
z30=4.3887*um
omrf0=36*2*pi*(10**6)/seconds
deltax=30*um#half the distance between the two DC electrode lines
Deltax=20*um#the width of a DC electrode
VRF0=250*eV#RF electrodes potential
RFw=110*um#This is the width of the RF electrode. This number I only have a rough estimate of, but it's prob not important
deltaz=35*um#This is half the length of the DC electrodes.
omrf0=36*2*pi*(10**6)/seconds
AllTrapPots=np.array([ 0.     ,  0.     ,  0.     ,  0.     ,  1.44865,  1.44865,
        1.44865,  1.44865,  1.44865,  1.44865,  1.44865,  9.20943,
        0.39616, -0.12035,  0.39616,  9.20943,  1.44865,  1.44865,
        1.44865,  1.44865,  1.44865,  1.44865,  1.44865, -0.21367,
        0.     ,  0.     ,  0.     ])*eV/2#half above and half below, hence the 1/2 factor







# Trap radius
r_0 = 0.5

# Initial dark matter conditions
a_old = np.array([0.0, 0.0, 0.0])
v_old = np.array([0.0, 0.0, 0.0])
#x as R[cosphicostheta, cosphisintheta,sinphi]

## test values
phi = np.pi / 4  # Example angle for phi (45 degrees)
theta = np.pi / 2 # Example angle for theta (90 degrees)


x_old = r_0 * np.array([
        np.cos(phi) * np.cos(theta),
        np.cos(phi) * np.sin(theta),
        np.sin(phi)
    ])

# Ions setup
ions = [
    np.array([0.3, 0.0, 0.0]),
    np.array([0.4, 0.0, 0.0]),
    np.array([0.5, 0.0, 0.0])
]
ion_velocities = [np.array([0.0, 0.0, 0.0]) for _ in ions]

# Data storage
position_of_particle = []
velocity_of_particle = []
acceleration_of_particle = []

#Store alpha values
alpha_vals = []  # store α_k(t) at each t

def compute_alpha_k(t,k,
    ions,      # list or array: E(t, r_i) for each ion i (assumed scalar or projection along mode axis)
    bik,      # 2D array: b_ik matrix, shape (N_ions, N_modes)
    omega_k,       # angular frequency of phonon mode k ASKKKK
    m_ion,         # mass of ion (kg)
    ):
    
    N = len(ions)
    assert bik.shape[0] == N
    
    # Scale factor: zero-point fluctuation
    scale = np.sqrt(1 / (2 * m_ion * hbar * omega_k))
    E_fields = [electric_field(epsilon, r_dm, ions) for ion in ions]
    #rdm is the rcharge aspect of the position of dark matter particle, so just position of particle
    
    # Sum over all ions
    total = 0.0 + 0.0j
    for i in range(N):
        E_i = E_fields[i]                    # electric field at ion i
        b_ik = ask                # phonon mode coupling coefficient
        total += E_i * b_ik * np.exp(1j * omega_k * t)
    
    alpha_k_t = 1j * elementary_charge * scale * total
    return alpha_k_t

def append_items(a, vnew, xnew):
    acceleration_of_particle.append(a)
    velocity_of_particle.append(vnew)
    position_of_particle.append(xnew)

def electric_field(epsilon, r_charge, ion):
    q = epsilon * elementary_charge
    r = ion - r_charge
    magnitude = np.linalg.norm(r)
    if magnitude == 0:
        return np.array([0.0, 0.0, 0.0])
    r_hat = r / magnitude
    return k_coulomb * (q / magnitude**2) * r_hat

def total_e(epsilon, r_charge, ions):
    total_field = np.zeros(3)
    for ion in ions:
        total_field += electric_field(epsilon, r_charge, ion)
    return total_field

def trap_force(x):
     return -trap_coefficient * x

def update_ions(x_dm):
    for i in range(len(ions)):
        E_ion = total_e(1, ions[i], [x_dm])
        F_ion = elementary_charge * E_ion
        F_trap = trap_force(ions[i])
        F_Total = F_ion + F_trap
        a_ion = F_Total / mass_dm
        ion_velocities[i] += a_ion * step
        ions[i] += ion_velocities[i] * step

# Main simulation loop
terminate = False
while not terminate:
    # Compute forces and update motion
    E_rt = total_e(1, x_old, ions)
    Fdm_electric = elementary_charge * E_rt
    Fdm_trap = trap_force(x_old)
    a = (Fdm_electric + Fdm_trap) / mass_dm
    vnew = v_old + a * step
    xnew = x_old + vnew * step
    append_items(a, vnew, xnew)

    # Update ions if inside trap radius
    if np.linalg.norm(x_old) <= r_0:
        update_ions(x_old)

    # Advance time and update state
    x_old, v_old, a_old = xnew, vnew, a
    t += step

    # Termination condition
    if t > time_limit or np.linalg.norm(xnew) > 10 * r_0:
        terminate = True
print(position_of_particle,velocity_of_particle,acceleration_of_particle)
print(ions)

#itai's functions

def divatan(up,down):
    #This is d(arctan2(up,down))/ddown up to a minus sign. It's useful for the pseudo-potential
    return up/(up**2+down**2)
def divatanup(up,down):
    #This is d(divatan(up,down))/dup
    return (down**2-up**2)/(up**2+down**2)**2
def divatandown(up,down):
    #This is d(divatan(up,down))/ddown
    return -2*up*down/(up**2+down**2)**2

def FRF(x,y,z,m=c.m,q=c.e,omrf=c.omega,VRF=c.vrf,ymin=c.y11,yedge1=c.y21,yedge2=c.y12,ymax=c.y22):
    #this is the pseudo potential, which is Z^2*(Div[PhiRF]/cos(om*t))^2/(4m*omega^2)
    divypart=divatan(z,yedge2-y)-divatan(z,yedge1-y)+divatan(z,ymin-y)-divatan(z,ymax-y)
    divzpart=divatan(yedge2-y,z)-divatan(yedge1-y,z)+divatan(ymin-y,z)-divatan(ymax-y,z) 
    divyparty=-2*divypart*(divatandown(z,yedge2-y)-divatandown(z,yedge1-y)+divatandown(z,ymin-y)-divatandown(z,ymax-y)) 
    divypartz=2*divypart*(divatanup(z,yedge2-y)-divatanup(z,yedge1-y)+divatanup(z,ymin-y)-divatanup(z,ymax-y)) 
    divzparty=-2*divzpart*(divatanup(yedge2-y,z)-divatanup(yedge1-y,z)+divatanup(ymin-y,z)-divatanup(ymax-y,z)) 
    divzpartz=2*divzpart*(divatandown(yedge2-y,z)-divatandown(yedge1-y,z)+divatandown(ymin-y,z)-divatandown(ymax-y,z)) 
    return -1*((VRF*q)/(2*pi*np.sqrt(m)*omrf))**2*np.array([divypartz*0,divzparty+divyparty,divzpartz+divypartz])


def anatangrad(xi,yi,x,y,z):
    #I just took a gradient of the above term. I don't really have a good explanation for why this is the result.
    dy=y-yi
    dx=x-xi
    r = math.sqrt(dx**2+dy**2+z**2); # added distance
    dry2=z**2+dy**2
    drx2=z**2+dx**2
    divy=z*dx/(r*dry2); # divide by factor r
    divz=-dy*dx*(1/dry2+1/drx2)/r; # divide by factor r
    divx=z*dy/(r*drx2); # divide by factor r
    return np.array([divx,divy,divz])

def FDC(x1,x2,z1,z2,x,y,z,q,VDC):
    #q=charge, VDC=voltage, 
    #limits of electrode are x1,x2 and z1,z2. x,y,z are coordinates of object
    return -q*(VDC/(2*pi))*(anatangrad(x2,z2,x,z,y)-anatangrad(x1,z2,x,z,y)
                         -anatangrad(x2,z1,x,z,y)+anatangrad(x1,z1,x,z,y))

def AllDCFs(x,y,z,q=1,VDC=AllTrapPots,deltax=deltax,deltaz=deltaz,Deltax=Deltax):
    upperones=0
    lowerones=0
    for aloc in range(len(VDC)):
        VDC0=VDC[aloc]
        upperones=upperones+FDC(deltax,deltax+Deltax,-len(VDC)*deltaz+2*deltaz*aloc,-len(VDC)*deltaz+deltaz*(2*aloc+2),x,y,z,q=q,VDC=VDC0)#+phiDC(deltax,deltax+Deltax,-3*deltaz,-deltaz,x,y,z,q,VDC=-VDC0)+phiDC(deltax,deltax+Deltax,deltaz,3*deltaz,x,y,z,q,VDC=-VDC0)
        lowerones=lowerones+FDC(-deltax-Deltax,-deltax,-len(VDC)*deltaz+2*deltaz*aloc,-len(VDC)*deltaz+deltaz*(2*aloc+2),x,y,z,q=q,VDC=VDC0)#+phiDC(-deltax-Deltax,-deltax,-3*deltaz,-deltaz,x,y,z,q,VDC=-VDC0)+phiDC(-deltax-Deltax,-deltax,deltaz,3*deltaz,x,y,z,q,VDC=-VDC0)
    return upperones+lowerones

def DMForce(x,y,z,q,m,x1=x10,y1=y10,z1=z10,x2=x20,y2=y20,z2=z20,x3=x30,y3=y30,z3=z30,omrf=omrf0,VRF=VRF0,VDC=AllTrapPots,
                       xmin=-deltax-Deltax-RFw,xedge1=-deltax-Deltax,xedge2=deltax+Deltax,xmax=deltax+Deltax+RFw,
                       deltax=deltax,deltaz=deltaz,Deltax=Deltax,pseudo=True):
    #with mirrors means to include the mirror charges
    Ftot=0
    if(pseudo):
        Ftot=Ftot+FRF(x,y,z,m=m,q=q,omrf=omrf,VRF=VRF,xmin=xmin,xedge1=xedge1,xedge2=xedge2,xmax=xmax)
    Ftot=Ftot+AllDCFs(x,y,z,q=q,VDC=VDC,deltax=deltax,deltaz=deltaz,Deltax=Deltax)
    for whichion in range(3):
        dx=x-[x1,x2,x3][whichion]
        dy=y-[y1,y2,y3][whichion]
        dz=z-[z1,z2,z3][whichion]
        Fpt0=(q*alpha/np.sqrt(dx**2+dy**2+dz**2)**3)
#         print('note that if x,y,z are scalars I may get an error/unexpected behavior')
        Ftot=Ftot+Fpt0[None]*np.array([dx,dy,dz])
    return Ftot

print(FRF(1*um,0,70*um))
print(FRF(1*um,1*um,70*um))
print(FRF(1*um,1*um,71*um))
print(FRF(1*um,-1*um,69*um))
print(FRF(1*um,-1*um,71*um))
print(FRF(1*um,1*um,90*um))
print(FRF(1*um,-1*um,60*um))
print(FRF(1*um,1*um,60*um))
print(FRF(1*um,-1*um,90*um))

[array([-2.16489014e+01, -3.53553391e+17, -3.53553391e+17])] [array([-2.16489014e+07, -3.53553391e+23, -3.53553391e+23])] [array([-2.16489014e+13, -3.53553391e+29, -3.53553391e+29])]
[array([-3.00000000e+17,  4.11362518e-13,  4.11362518e-13]), array([-4.00000000e+17,  3.10647104e-13,  3.10647104e-13]), array([-5.00000000e+17,  2.30668392e-13,  2.30668392e-13])]
[-0.00000000e+00 -0.00000000e+00  5.22582731e-31]
[0.00000000e+00 7.46088972e-33 5.22262914e-31]
[0.00000000e+00 6.85229799e-33 4.86513730e-31]
[ 0.00000000e+00 -8.13348213e-33  5.61210967e-31]
[ 0.00000000e+00 -6.85229799e-33  4.86513730e-31]
[0.00000000e+00 1.65207345e-33 1.48686719e-31]
[ 0.00000000e+00 -1.88093977e-32  1.12856572e-30]
[0.00000000e+00 1.88093977e-32 1.12856572e-30]
[ 0.00000000e+00 -1.65207345e-33  1.48686719e-31]
